# MindTrace - Advanced Cognitive Pattern Analyzer

## Advanced AI-Powered Cognitive Analysis System

### 1. Setup and Imports

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import re
import warnings
warnings.filterwarnings('ignore')

# NLP Libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import spacy

# ML Libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import LatentDirichletAllocation, PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.linear_model import LinearRegression
from sklearn.metrics import silhouette_score
from collections import Counter

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud

# Download NLTK data
for pkg in ["stopwords", "punkt", "wordnet", "punkt_tab", "vader_lexicon"]:
    try: nltk.download(pkg, quiet=True)
    except: pass

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Initialize
sia = SentimentIntensityAnalyzer()
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

# Configure
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

print("All libraries loaded successfully!")

### 2. Data Ingestion Module

In [ ]:
class DataIngestion:
    def __init__(self):
        self.data = None
        self.metadata = {}
        
    def load_csv(self, filepath, timestamp_col="timestamp", text_col="text", delimiter=","):
        df = pd.read_csv(filepath, delimiter=delimiter)
        date_formats = ["%Y-%m-%d %H:%M:%S", "%Y-%m-%d", "%d/%m/%Y %H:%M:%S", "%m/%d/%Y %H:%M:%S"]
        for fmt in date_formats:
            try:
                df[timestamp_col] = pd.to_datetime(df[timestamp_col], format=fmt)
                break
            except: continue
        else:
            df[timestamp_col] = pd.to_datetime(df[timestamp_col], infer_datetime_format=True)
        df = df.rename(columns={timestamp_col: "timestamp", text_col: "text"})
        df = df[["timestamp", "text"]].dropna()
        df = df.sort_values("timestamp").reset_index(drop=True)
        self.data = df
        self.metadata = {"source": filepath, "rows": len(df)}
        return df
    
    def generate_sample_data(self, n_entries=1000, start_date="2024-01-01"):
        np.random.seed(42)
        texts = {
            "work": ["Working on project proposal", "Code review completed", "Meeting with team", "Bug fix released", "Documentation updated", "Refactoring code", "Client call went well"],
            "stress": ["Feeling overwhelmed", "Stressed about deadline", "Cannot focus", "Everything is going wrong", "Feeling anxious"],
            "achievement": ["Just finished milestone", "Received great feedback", "Solved complex problem", "Productivity is high", "Feeling accomplished"],
            "personal": ["Great dinner with family", "Watched movie", "Practicing meditation", "Reading book", "Grateful for everything"],
            "learning": ["Learned new concept", "Exploring framework", "Reading research paper", "Completed course module", "Experimenting with tools"],
            "casual": ["Just checking in", "Random thoughts", "Not much happening", "Watching TV", "Relaxing evening"]
        }
        
        start = pd.to_datetime(start_date)
        hour_weights = [0.02, 0.01, 0.01, 0.01, 0.02, 0.03, 0.05, 0.08, 0.12, 0.10, 0.08, 0.06, 0.07, 0.08, 0.10, 0.08, 0.05, 0.04, 0.03, 0.02, 0.02, 0.02, 0.01, 0.01]
        
        data = []
        for _ in range(n_entries):
            hour = np.random.choice(24, p=hour_weights)
            day_offset = np.random.randint(0, 90)
            ts = start + timedelta(days=day_offset, hours=hour, minutes=np.random.randint(0,60))
            
            if 9 <= hour <= 17:
                cat = np.random.choice(["work", "work", "work", "stress", "achievement", "learning"], p=[0.3, 0.2, 0.15, 0.15, 0.1, 0.1])
            else:
                cat = np.random.choice(["personal", "casual", "learning"], p=[0.5, 0.3, 0.2])
            
            text = np.random.choice(texts[cat])
            data.append({"timestamp": ts, "text": text})
        
        df = pd.DataFrame(data).sort_values("timestamp").reset_index(drop=True)
        self.data = df
        return df

ingestion = DataIngestion()
print("DataIngestion ready!")

### 3. Advanced Text Preprocessing

In [ ]:
class TextPreprocessor:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        
    def clean(self, text):
        text = text.lower()
        text = re.sub(r"http\S+|www\S+", "", text)
        text = re.sub(r"[^a-zA-Z\s]", "", text)
        return " ".join(text.split())
    
    def remove_stop(self, text):
        words = text.split()
        return " ".join([w for w in words if w not in stop_words])
    
    def lemmatize(self, text):
        return " ".join([lemmatizer.lemmatize(w) for w in text.split()])
    
    def extract_entities(self, text):
        doc = self.nlp(text)
        return {"persons": [e.text for e in doc.ents if e.label_ == "PERSON"],
                "orgs": [e.text for e in doc.ents if e.label_ == "ORG"],
                "dates": [e.text for e in doc.ents if e.label_ == "DATE"]}
    
    def process(self, df):
        df = df.copy()
        df["cleaned"] = df["text"].apply(self.clean)
        df["no_stop"] = df["cleaned"].apply(self.remove_stop)
        df["processed"] = df["no_stop"].apply(self.lemmatize)
        df["entities"] = df["text"].apply(self.extract_entities)
        return df

preprocessor = TextPreprocessor()
print("TextPreprocessor ready!")

### 4. Advanced NLP Analysis

In [ ]:
class NLPAnalyzer:
    def __init__(self):
        self.emotions = {
            "joy": ["happy", "joy", "excited", "love", "great", "wonderful", "amazing", "grateful", "proud", "motivated"],
            "anger": ["angry", "furious", "frustrated", "annoyed", "hate", "terrible", "awful", "stupid"],
            "sadness": ["sad", "depressed", "unhappy", "lonely", "disappointed", "grief", "hopeless", "tired"],
            "fear": ["afraid", "scared", "anxious", "worried", "nervous", "panic", "stress", "overwhelmed"],
            "surprise": ["surprised", "shocked", "amazed", "astonished", "unexpected", "stunned"],
            "anticipation": ["waiting", "hope", "expect", "looking forward", "planned", "upcoming", "goal"]
        }
    
    def sentiment_vader(self, text):
        scores = sia.polarity_scores(text)
        compound = scores["compound"]
        label = "positive" if compound > 0.05 else "negative" if compound < -0.05 else "neutral"
        return {"polarity": compound, "label": label, "pos": scores["pos"], "neg": scores["neg"]}
    
    def sentiment_blob(self, text):
        from textblob import TextBlob
        blob = TextBlob(text)
        return {"polarity": blob.sentiment.polarity, "subjectivity": blob.sentiment.subjectivity}
    
    def detect_emotions(self, text):
        text_lower = text.lower()
        scores = {}
        for emotion, keywords in self.emotions.items():
            scores[emotion] = sum(1 for kw in keywords if kw in text_lower)
        max_score = max(scores.values()) if max(scores.values()) > 0 else 1
        scores = {k: v/max_score for k, v in scores.items()}
        return {"emotions": scores, "dominant": max(scores, key=scores.get)}
    
    def process(self, df):
        df = df.copy()
        vader = df["text"].apply(self.sentiment_vader)
        df["vader_polarity"] = vader.apply(lambda x: x["polarity"])
        df["sentiment_label"] = vader.apply(lambda x: x["label"])
        
        blob = df["text"].apply(self.sentiment_blob)
        df["polarity"] = blob.apply(lambda x: x["polarity"])
        df["subjectivity"] = blob.apply(lambda x: x["subjectivity"])
        
        emo = df["text"].apply(self.detect_emotions)
        for e in self.emotions:
            df[f"emotion_{e}"] = emo.apply(lambda x: x["emotions"].get(e, 0))
        df["emotion_dominant"] = emo.apply(lambda x: x["dominant"])
        
        return df

nlp_analyzer = NLPAnalyzer()
print("NLPAnalyzer ready!")

### 5. Cognitive Pattern Detection

In [ ]:
class PatternDetector:
    def add_time_features(self, df):
        df = df.copy()
        df["hour"] = df["timestamp"].dt.hour
        df["day_of_week"] = df["timestamp"].dt.dayofweek
        df["day_name"] = df["timestamp"].dt.day_name()
        df["month"] = df["timestamp"].dt.month
        df["week"] = df["timestamp"].dt.isocalendar().week
        df["date"] = df["timestamp"].dt.date
        df["is_weekend"] = df["day_of_week"].isin([5, 6])
        
        def period(h):
            if h < 9: return "early"
            elif h < 12: return "morning"
            elif h < 14: return "midday"
            elif h < 17: return "afternoon"
            elif h < 21: return "evening"
            else: return "night"
        
        df["time_period"] = df["hour"].apply(period)
        return df
    
    def productivity_windows(self, df):
        hourly = df.groupby("hour").agg({"polarity": "mean", "text": "count"})
        hourly["score"] = hourly["polarity"] * 0.5 + (hourly["text"] / hourly["text"].max()) * 0.5
        peaks = hourly.nlargest(3, "score").index.tolist()
        return {"peak_hours": peaks, "hourly_stats": hourly}
    
    def behavioral_trends(self, df):
        df = df.sort_values("timestamp").reset_index(drop=True)
        df["idx"] = range(len(df))
        
        X = df[["idx"]].values
        y = df["polarity"].values
        
        model = LinearRegression()
        model.fit(X, y)
        slope = model.coef_[0]
        
        trend = "improving" if slope > 0.001 else "declining" if slope < -0.001 else "stable"
        
        focus_kw = ["work", "task", "project", "goal", "finish", "deadline"]
        dist_kw = ["browse", "social", "watch", "game", "chat"]
        
        def focus_score(text):
            f = sum(1 for k in focus_kw if k in text.lower())
            d = sum(1 for k in dist_kw if k in text.lower())
            return f / (f + d) if f + d > 0 else 0.5
        
        df["focus_score"] = df["text"].apply(focus_score)
        
        return {"sentiment_trend": trend, "slope": slope, "focus_score": df["focus_score"].mean()}
    
    def process(self, df):
        df = self.add_time_features(df)
        prod = self.productivity_windows(df)
        behav = self.behavioral_trends(df)
        return df, {"productivity": prod, "behavioral": behav}

pattern_detector = PatternDetector()
print("PatternDetector ready!")

### 6. Time Series Analysis

In [ ]:
class TimeSeriesAnalyzer:
    def aggregate(self, df, period="D"):
        df = df.set_index("timestamp")
        agg = df.resample(period).agg({"polarity": ["mean", "std", "min", "max"], "text": "count"})
        agg.columns = ["_".join(c) for c in agg.columns]
        return agg.reset_index()
    
    def detect_trend(self, df):
        daily = self.aggregate(df)
        if len(daily) < 7:
            return {"trend": "insufficient_data"}
        
        y = daily["polarity_mean"].dropna().values
        x = np.arange(len(y))
        
        try:
            from scipy import stats
            slope, intercept, r, p, se = stats.linregress(x, y)
        except:
            model = LinearRegression()
            model.fit(x.reshape(-1,1), y)
            slope, r = model.coef_[0], 0
        
        direction = "increasing" if slope > 0.001 else "decreasing" if slope < -0.001 else "stable"
        return {"direction": direction, "slope": slope}
    
    def detect_cycles(self, df):
        hourly = df.groupby("hour").size()
        dow = df.groupby("day_of_week").size()
        return {"hourly_pattern": hourly.to_dict(), "daily_pattern": dow.to_dict()}
    
    def process(self, df):
        return {"trend": self.detect_trend(df), "cycles": self.detect_cycles(df)}

ts_analyzer = TimeSeriesAnalyzer()
print("TimeSeriesAnalyzer ready!")

### 7. Semantic Clustering

In [ ]:
class ClusterAnalyzer:
    def create_embeddings(self, texts):
        vec = TfidfVectorizer(max_features=500, stop_words="english", ngram_range=(1,2))
        emb = vec.fit_transform(texts)
        return emb, vec
    
    def cluster(self, embeddings, n_clusters=4):
        km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = km.fit_predict(embeddings)
        return labels, km
    
    def analyze_clusters(self, df, labels):
        df = df.copy()
        df["cluster"] = labels
        
        analysis = {}
        for c in df["cluster"].unique():
            cd = df[df["cluster"] == c]
            pol = cd["polarity"].mean()
            sub = cd["subjectivity"].mean()
            foc = cd["focus_score"].mean() if "focus_score" in cd.columns else 0.5
            
            if pol > 0.2 and foc > 0.6:
                label = "Deep Work"
            elif pol < -0.1:
                label = "Stress-Driven"
            elif sub > 0.5:
                label = "Reflective"
            else:
                label = "Analytical"
            
            analysis[c] = {"label": label, "size": len(cd), "polarity": pol, "subjectivity": sub}
        
        return analysis
    
    def reduce_dims(self, embeddings):
        svd = TruncatedSVD(n_components=min(50, embeddings.shape[1]-1), random_state=42)
        reduced = svd.fit_transform(embeddings)
        
        tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)-1))
        return tsne.fit_transform(reduced)
    
    def process(self, df):
        texts = df["processed"].fillna("").tolist()
        emb, vec = self.create_embeddings(texts)
        labels, _ = self.cluster(emb)
        
        df = df.copy()
        df["cluster"] = labels
        
        coords = self.reduce_dims(emb.toarray())
        df["x"] = coords[:, 0]
        df["y"] = coords[:, 1]
        
        analysis = self.analyze_clusters(df, labels)
        profile = {"dominant": max(analysis.items(), key=lambda x: x[1]["size"])[1]["label"]}
        
        return df, analysis, profile

cluster_analyzer = ClusterAnalyzer()
print("ClusterAnalyzer ready!")

### 8. Interactive Visualizations

In [ ]:
class Visualizer:
    def __init__(self):
        self.colors = {"positive": "#2ecc71", "neutral": "#3498db", "negative": "#e74c3c",
                      "joy": "#f1c40f", "anger": "#e74c3c", "sadness": "#9b59b6", 
                      "fear": "#e67e22", "surprise": "#1abc9c"}
    
    def sentiment_timeline(self, df):
        daily = df.groupby("date").agg({"polarity": "mean", "text": "count"}).reset_index()
        daily.columns = ["date", "polarity", "count"]
        
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=daily["date"], y=daily["polarity"], mode="lines+markers",
                                name="Sentiment", line=dict(color=self.colors["positive"], width=2)))
        daily["ma7"] = daily["polarity"].rolling(7, min_periods=1).mean()
        fig.add_trace(go.Scatter(x=daily["date"], y=daily["ma7"], mode="lines", name="7-day MA",
                                line=dict(color="#95a5a6", dash="dash")))
        fig.update_layout(title="Sentiment Timeline", xaxis_title="Date", yaxis_title="Polarity",
                         template="plotly_white", height=400)
        fig.add_hline(y=0, line_dash="dot", line_color="gray")
        return fig
    
    def emotion_bar(self, df):
        emotions = ["joy", "anger", "sadness", "fear", "surprise", "anticipation"]
        values = [df[f"emotion_{e}"].mean() for e in emotions]
        
        fig = go.Figure(go.Bar(x=emotions, y=values, marker_color=[self.colors.get(e, "#3498db") for e in emotions]))
        fig.update_layout(title="Emotion Distribution", xaxis_title="Emotion", yaxis_title="Score",
                         template="plotly_white", height=400)
        return fig
    
    def sentiment_pie(self, df):
        counts = df["sentiment_label"].value_counts()
        fig = go.Figure(go.Pie(labels=counts.index, values=counts.values, hole=0.4,
                              marker=dict(colors=[self.colors.get(c, "#95a5a6") for c in counts.index])))
        fig.update_layout(title="Sentiment Breakdown", template="plotly_white", height=400)
        return fig
    
    def activity_heatmap(self, df):
        heatmap = df.groupby(["day_of_week", "hour"]).size().unstack(fill_value=0)
        heatmap = heatmap.reindex([0,1,2,3,4,5,6])
        heatmap.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
        
        fig = go.Figure(go.Heatmap(z=heatmap.values, x=list(range(24)), y=heatmap.index,
                                   colorscale="Blues"))
        fig.update_layout(title="Activity Heatmap", xaxis_title="Hour", yaxis_title="Day",
                         template="plotly_white", height=400)
        return fig
    
    def clusters_scatter(self, df):
        if "x" not in df.columns:
            return None
        fig = px.scatter(df, x="x", y="y", color="cluster", hover_data=["text"],
                        title="Thinking Style Clusters", template="plotly_white", height=500)
        return fig
    
    def wordcloud(self, df):
        text = " ".join(df["processed"].fillna("").tolist())
        wc = WordCloud(width=800, height=400, background_color="white", max_words=50).generate(text)
        
        fig = go.Figure()
        fig.add_layout_image(source=wc.to_image(), x=0, y=1, xref="paper", yref="paper",
                           sizex=1, sizey=1, sizing="stretch")
        fig.update_layout(title="Word Cloud", xaxis=dict(visible=False), yaxis=dict(visible=False),
                         height=400, margin=dict(l=0,r=0,t=40,b=0))
        return fig
    
    def hourly_productivity(self, df):
        hourly = df.groupby("hour").agg({"polarity": "mean", "focus_score": "mean"}).reset_index()
        
        fig = make_subplots(specs=[[{"secondary_y": True}]])
        fig.add_trace(go.Bar(x=hourly["hour"], y=hourly["focus_score"], name="Activity",
                            marker_color="#3498db"), secondary_y=False)
        fig.add_trace(go.Scatter(x=hourly["hour"], y=hourly["polarity"], name="Sentiment",
                               line=dict(color="#e74c3c", width=3)), secondary_y=True)
        fig.update_layout(title="Hourly Productivity", template="plotly_white", height=400)
        return fig

visualizer = Visualizer()
print("Visualizer ready!")

### 9. Insight Generation Engine

In [ ]:
class InsightGenerator:
    def generate(self, df, patterns, ts_results, cluster_profile):
        insights = {"productivity": [], "emotional": [], "behavioral": [], "recommendations": []}
        
        # Productivity insights
        if patterns.get("productivity", {}).get("peak_hours"):
            peaks = patterns["productivity"]["peak_hours"]
            if peaks:
                insights["productivity"].append(f"Your most productive hours are {peaks[0]}:00-{peaks[0]+1}:00")
        
        # Emotional insights
        avg_pol = df["polarity"].mean()
        if avg_pol > 0.2:
            insights["emotional"].append("You maintain a positive emotional outlook overall")
        elif avg_pol < -0.1:
            insights["emotional"].append("Recent entries show negative sentiment - consider self-care")
        else:
            insights["emotional"].append("Your emotional balance is relatively neutral")
        
        # Dominant emotion
        dom = df["emotion_dominant"].mode().iloc[0]
        insights["emotional"].append(f"Dominant emotion: {dom.title()}")
        
        # Behavioral insights
        if patterns.get("behavioral", {}).get("sentiment_trend"):
            trend = patterns["behavioral"]["sentiment_trend"]
            if trend == "improving":
                insights["behavioral"].append("Sentiment has been improving over time")
            elif trend == "declining":
                insights["behavioral"].append("Sentiment has been declining recently")
        
        # Focus
        if patterns.get("behavioral", {}).get("focus_score"):
            foc = patterns["behavioral"]["focus_score"]
            if foc > 0.6:
                insights["productivity"].append("High focus score - great concentration!")
            elif foc < 0.4:
                insights["recommendations"].append("Consider improving focus with time-blocking")
        
        # Time series
        if ts_results.get("trend", {}).get("direction"):
            direction = ts_results["trend"]["direction"]
            if direction == "increasing":
                insights["behavioral"].append("Positive trend detected in recent days")
        
        # Recommendations
        neg_ratio = (df["sentiment_label"] == "negative").mean()
        if neg_ratio > 0.3:
            insights["recommendations"].append("High negative sentiment - try daily gratitude journaling")
        
        if df["focus_score"].mean() < 0.4:
            insights["recommendations"].append("Focus is low - reduce distractions during work hours")
        
        return insights

insight_generator = InsightGenerator()
print("InsightGenerator ready!")

### 10. Cognitive Report Generator

In [ ]:
class ReportGenerator:
    def generate(self, df, insights, patterns, cluster_analysis, cluster_profile):
        report = f"""
============================================================
         MINDTRACE - COGNITIVE ANALYSIS REPORT
============================================================

DATA SUMMARY
------------------------------------------------------------
Total Entries: {len(df):,}
Date Range: {df['timestamp'].min().strftime('%Y-%m-%d')} to {df['timestamp'].max().strftime('%Y-%m-%d')}
Days Analyzed: {(df['timestamp'].max() - df['timestamp'].min()).days}

SENTIMENT ANALYSIS
------------------------------------------------------------
Average Sentiment: {df['polarity'].mean():.3f} ({df['sentiment_label'].mode().iloc[0]})
Sentiment Std Dev: {df['polarity'].std():.3f}
Positive: {(df['sentiment_label'] == 'positive').mean()*100:.1f}%
Neutral: {(df['sentiment_label'] == 'neutral').mean()*100:.1f}%
Negative: {(df['sentiment_label'] == 'negative').mean()*100:.1f}%

EMOTION PROFILE
------------------------------------------------------------
Dominant Emotion: {df['emotion_dominant'].mode().iloc[0].title()}
Joy Score: {df['emotion_joy'].mean():.3f}
Sadness Score: {df['emotion_sadness'].mean():.3f}
Fear Score: {df['emotion_fear'].mean():.3f}
Anger Score: {df['emotion_anger'].mean():.3f}

KEY INSIGHTS
------------------------------------------------------------
"""
        for category, items in insights.items():
            if items:
                report += f"
{category.upper()}:
"
                for item in items[:3]:
                    report += f"  - {item}
"
        
        report += f"""
THINKING STYLE ANALYSIS
------------------------------------------------------------
"""
        if cluster_profile:
            report += f"Dominant Style: {cluster_profile.get('dominant', 'N/A')}
"
        
        if cluster_analysis:
            report += "
Style Distribution:
"
            for cid, info in cluster_analysis.items():
                pct = info['size'] / len(df) * 100
                report += f"  - {info['label']}: {pct:.1f}%
"
        
        report += "
============================================================
"
        report += "                  END OF REPORT
"
        report += "============================================================"
        
        return report
    
    def display(self, df, insights, patterns, cluster_analysis, cluster_profile):
        report = self.generate(df, insights, patterns, cluster_analysis, cluster_profile)
        print(report)
        return report

report_generator = ReportGenerator()
print("ReportGenerator ready!")

### 11. Main Dashboard - Run Complete Analysis

In [ ]:
class MindTraceDashboard:
    def __init__(self):
        self.df = None
        self.results = {}
        
    def run_analysis(self, n_samples=1000):
        print("="*60)
        print("   MINDTRACE - COGNITIVE PATTERN ANALYZER")
        print("="*60)
        print("\nStep 1: Generating sample data...")
        self.df = ingestion.generate_sample_data(n_entries=n_samples)
        print(f"   Generated {len(self.df)} entries")
        
        print("\nStep 2: Preprocessing text...")
        self.df = preprocessor.process(self.df)
        
        print("\nStep 3: NLP Analysis...")
        self.df = nlp_analyzer.process(self.df)
        
        print("\nStep 4: Pattern Detection...")
        self.df, patterns = pattern_detector.process(self.df)
        self.results["patterns"] = patterns
        
        print("\nStep 5: Time Series Analysis...")
        self.results["time_series"] = ts_analyzer.process(self.df)
        
        print("\nStep 6: Clustering Analysis...")
        self.df, cluster_analysis, cluster_profile = cluster_analyzer.process(self.df)
        self.results["clusters"] = cluster_analysis
        self.results["profile"] = cluster_profile
        
        print("\nStep 7: Generating Insights...")
        self.results["insights"] = insight_generator.generate(self.df, patterns, 
                                                             self.results["time_series"], cluster_profile)
        
        print("\n" + "="*60)
        print("   ANALYSIS COMPLETE!")
        print("="*60)
        
        return self.df
    
    def show_visualizations(self):
        print("\nDisplaying Visualizations...")
        
        fig = visualizer.sentiment_timeline(self.df)
        fig.show()
        
        fig = visualizer.emotion_bar(self.df)
        fig.show()
        
        fig = visualizer.sentiment_pie(self.df)
        fig.show()
        
        fig = visualizer.activity_heatmap(self.df)
        fig.show()
        
        fig = visualizer.clusters_scatter(self.df)
        if fig:
            fig.show()
        
        fig = visualizer.wordcloud(self.df)
        fig.show()
        
        fig = visualizer.hourly_productivity(self.df)
        fig.show()
    
    def generate_report(self):
        report_generator.display(self.df, self.results["insights"], self.results["patterns"],
                              self.results["clusters"], self.results["profile"])

dashboard = MindTraceDashboard()
print("MindTraceDashboard ready!")

### 12. Execute Analysis

In [ ]:
# Run complete analysis
dashboard.run_analysis(n_samples=1000)

### 13. Display Visualizations

In [ ]:
# Show all visualizations
dashboard.show_visualizations()

### 14. Generate Report

In [ ]:
# Generate cognitive report
dashboard.generate_report()